In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from AlignmentTechniques import LatentAlignment2d, AdaptiveBatchNorm2d, EuclideanAlignment

class SeizureConvNet(nn.Module):
    """
    Deep Convolutional Neural Network ispirata alle architetture Deep4Net 
    ottimizzate per l'estrazione di anomalie ad alta frequenza (Seizure Detection).
    """
    def __init__(self, in_shape, n_out, alignment='None', dropout=0.5):
        super(SeizureConvNet, self).__init__()
        self.in_shape = in_shape
        self.n_out = n_out
        self.alignment = alignment

        # Numero di filtri di base che raddoppia ad ogni strato
        n_filters = 25

        if alignment == 'euclidean':
            self.euclidean = EuclideanAlignment()

        # Input BatchNorm / Latent Alignment
        if alignment == 'latent':
            self.latent_align0 = LatentAlignment2d(in_shape[0], affine=False)
        elif alignment == 'adaptive':
            self.abn0 = AdaptiveBatchNorm2d(in_shape[0], affine=False)
        else:
            self.bn0 = nn.BatchNorm2d(in_shape[0], affine=False)

        # --- BLOCCO 1: Estrazione Temporale-Spaziale ---
        self.conv1_time = nn.Conv2d(1, n_filters, kernel_size=(1, 11), padding=(0, 5), bias=False)
        self.conv1_space = nn.Conv2d(n_filters, n_filters, kernel_size=(in_shape[0], 1), bias=False)
        if alignment == 'latent':
            self.latent_align1 = LatentAlignment2d(n_filters)
        elif alignment == 'adaptive':
            self.abn1 = AdaptiveBatchNorm2d(n_filters)
        else:
            self.bn1 = nn.BatchNorm2d(n_filters)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop1 = nn.Dropout(dropout)

        # --- BLOCCO 2: Estrazione Profonda ---
        self.conv2 = nn.Conv2d(n_filters, n_filters * 2, kernel_size=(1, 11), padding=(0, 5), bias=False)
        if alignment == 'latent':
            self.latent_align2 = LatentAlignment2d(n_filters * 2)
        elif alignment == 'adaptive':
            self.abn2 = AdaptiveBatchNorm2d(n_filters * 2)
        else:
            self.bn2 = nn.BatchNorm2d(n_filters * 2)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop2 = nn.Dropout(dropout)

        # --- BLOCCO 3: Feature di alto livello ---
        self.conv3 = nn.Conv2d(n_filters * 2, n_filters * 4, kernel_size=(1, 11), padding=(0, 5), bias=False)
        if alignment == 'latent':
            self.latent_align3 = LatentAlignment2d(n_filters * 4)
        elif alignment == 'adaptive':
            self.abn3 = AdaptiveBatchNorm2d(n_filters * 4)
        else:
            self.bn3 = nn.BatchNorm2d(n_filters * 4)
        self.pool3 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))
        self.drop3 = nn.Dropout(dropout)

        # --- CLASSIFICATORE ---
        # Calcolo dinamico della dimensione residua nel tempo dopo i 3 Max Pooling
        final_time = in_shape[1] // 3 // 3 // 3
        self.n_features = n_filters * 4 * final_time
        self.fc_out = nn.Linear(self.n_features, n_out)

    def forward(self, x, sbj_trials):
        """
        Args:
             x: (batch * sbj_trials, spatial, time)
             sbj_trials: number of trials per subject
        """
        _, spatial, time = x.shape

        if self.alignment == 'euclidean':
            x = self.euclidean(x, sbj_trials)

        x = x.reshape(-1, spatial, 1, time)
        if self.alignment == 'latent':
            x = self.latent_align0(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn0(x, sbj_trials)
        else:
            x = self.bn0(x)
        x = x.reshape(-1, spatial, time)

        x = x.unsqueeze(1) # Artificial image channel (batch, 1, spatial, time)

        # Blocco 1
        x = self.conv1_time(x)
        x = self.conv1_space(x)
        if self.alignment == 'latent':
            x = self.latent_align1(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn1(x, sbj_trials)
        else:
            x = self.bn1(x)
        x = F.elu(x)
        x = self.pool1(x)
        x = self.drop1(x)

        # Blocco 2
        x = self.conv2(x)
        if self.alignment == 'latent':
            x = self.latent_align2(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn2(x, sbj_trials)
        else:
            x = self.bn2(x)
        x = F.elu(x)
        x = self.pool2(x)
        x = self.drop2(x)

        # Blocco 3
        x = self.conv3(x)
        if self.alignment == 'latent':
            x = self.latent_align3(x, sbj_trials)
        elif self.alignment == 'adaptive':
            x = self.abn3(x, sbj_trials)
        else:
            x = self.bn3(x)
        x = F.elu(x)
        x = self.pool3(x)
        x = self.drop3(x)

        x = x.reshape(-1, self.n_features)
        x = self.fc_out(x)

        return x

In [2]:
import os
import urllib.request
import mne
import numpy as np
import pandas as pd
import gc

# 1. Mappatura manuale dei file e dei tempi delle crisi (start_s, end_s)
# Questo ci evita di dover scaricare e analizzare i complessi file di testo "summary"
epilepsy_data_map = {
    1: {'file': 'chb01_03.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb01/chb01_03.edf', 'seizures': [(2996, 3036)]},
    2: {'file': 'chb02_16.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb02/chb02_16.edf', 'seizures': [(130, 212)]},
    3: {'file': 'chb03_01.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb03/chb03_01.edf', 'seizures': [(362, 414)]},
    4: {'file': 'chb05_06.edf', 'url': 'https://physionet.org/files/chbmit/1.0.0/chb05/chb05_06.edf', 'seizures': [(417, 532)]}
}

def extract_epilepsy_data(subject_id, info):
    filename = info['file']
    # Scarica il file solo se non è già presente nella cartella
    if not os.path.exists(filename):
        print(f"   Scaricando {filename} da PhysioNet (circa 30-50MB)...")
        urllib.request.urlretrieve(info['url'], filename)

    # Caricamento del file raw
    raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)
    
    # Il CHB-MIT può avere canali leggermente diversi tra soggetti. 
    # Per garantire che i tensori combacino, prendiamo i primi 18 canali standard.
    raw.pick(raw.ch_names[:18], verbose=False)
    
    # Ricampionamento a 100 Hz per alleggerire la rete neurale e uniformare i dati
    raw.resample(100.0, verbose=False)

    # Creazione di epoche di 4 secondi (400 campioni a 100 Hz)
    epochs = mne.make_fixed_length_epochs(raw, duration=4.0, preload=True, verbose=False)
    X = epochs.get_data() * 1e6  # Conversione in microvolts
    
    # Inizializzazione delle etichette a 0 (Background / Nessuna crisi)
    y = np.zeros(len(X), dtype=np.int64)
    
    # I tempi di inizio di ogni epoca in secondi
    epoch_times = epochs.events[:, 0] / raw.info['sfreq'] 
    
    # Etichettatura: se l'epoca si sovrappone al periodo della crisi, diventa 1 (Target)
    for (start_sz, end_sz) in info['seizures']:
        for i, t in enumerate(epoch_times):
            if (t + 4.0) > start_sz and t < end_sz:
                y[i] = 1
                
    return X, y

# --- ESECUZIONE DELL'ESTRAZIONE ---
X_list, y_list, meta_list = [], [], []

print("Inizio Estrazione Dati Epilessia (CHB-MIT)...")
for sbj, info in epilepsy_data_map.items():
    print(f"Elaborazione Paziente {sbj}...")
    X_tmp, y_tmp = extract_epilepsy_data(sbj, info)
    
    X_list.append(X_tmp.astype('float32'))
    y_list.append(y_tmp)
    meta_list.append(pd.DataFrame({'subject': [sbj] * len(y_tmp)}))
    
    del X_tmp
    gc.collect()

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
metadata = pd.concat(meta_list, ignore_index=True)

print(f"\n✅ Caricamento e Preparazione Completati!")
print(f"Shape finale (X): {X.shape} -> (prove, 18 canali, 400 campioni)")

# Analizziamo lo sbilanciamento
n_seizures = np.sum(y == 1)
n_background = np.sum(y == 0)
print(f"\n--- ANALISI SBILANCIAMENTO ---")
print(f"Epoche Background (Classe 0): {n_background}")
print(f"Epoche Crisi (Classe 1): {n_seizures}")
print(f"Rapporto: circa {n_background // n_seizures} a 1!")

Inizio Estrazione Dati Epilessia (CHB-MIT)...
Elaborazione Paziente 1...
   Scaricando chb01_03.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 2...
   Scaricando chb02_16.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 3...
   Scaricando chb03_01.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)


Elaborazione Paziente 4...
   Scaricando chb05_06.edf da PhysioNet (circa 30-50MB)...


/tmp/ipykernel_18963/2751087627.py:25: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(filename, preload=True, verbose=False)



✅ Caricamento e Preparazione Completati!
Shape finale (X): (2939, 18, 400) -> (prove, 18 canali, 400 campioni)

--- ANALISI SBILANCIAMENTO ---
Epoche Background (Classe 0): 2865
Epoche Crisi (Classe 1): 74
Rapporto: circa 38 a 1!


In [3]:
import torch
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import balanced_accuracy_score, f1_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizzando il device: {device}")

X_tensor = torch.from_numpy(X).to(device)
y_tensor = torch.from_numpy(y).to(device)

n_channels = X.shape[1]
n_samples = X.shape[2]
n_classes = len(np.unique(y))

# --- CALCOLO DEI PESI PER LA LOSS SBILANCIATA ---
class_counts = np.bincount(y)
weights = 1.0 / torch.tensor(class_counts, dtype=torch.float32)
weights = weights / weights.sum() * 2.0 # Normalizzazione
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print(f"Pesi applicati alla Loss: Background={weights[0]:.4f}, Crisi={weights[1]:.4f}")

# --- TRAINING LOOP LOSO ---
print("\n=== Inizio Validazione LOSO su Epilessia (SeizureConvNet + Latent Alignment) ===")
loso_bal_acc = []
loso_f1 = []
subjects_array = metadata['subject'].unique()
num_epochs = 30 # Le Deep CNN hanno bisogno di qualche epoca in più

for test_subject in subjects_array:
    print(f"\n-> Addestramento per Test Subject: {test_subject}")
    
    train_mask = (metadata['subject'] != test_subject).values
    test_mask = (metadata['subject'] == test_subject).values
    
    model = SeizureConvNet(in_shape=(n_channels, n_samples), n_out=n_classes, alignment='latent').to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_subjects = metadata[train_mask]['subject'].unique()
    
    model.train()
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for subj in train_subjects:
            subj_mask = (metadata['subject'] == subj).values
            batch_X = X_tensor[subj_mask]
            batch_y = y_tensor[subj_mask]
            
            optimizer.zero_grad(set_to_none=True)
            
            # Deep Sets: passiamo tutto il soggetto
            outputs = model(batch_X, sbj_trials=batch_X.shape[0])
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        if (epoch + 1) % 10 == 0:
            print(f"   Epoca [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_subjects):.4f}")
            
    # --- VALUTAZIONE ---
    model.eval()
    with torch.no_grad():
        test_X = X_tensor[test_mask]
        test_y = y_tensor[test_mask]
        
        outputs = model(test_X, sbj_trials=test_X.shape[0])
        _, predicted = torch.max(outputs.data, 1)
        
        y_true = test_y.cpu().numpy()
        y_pred = predicted.cpu().numpy()
        
        # Calcolo delle metriche robuste
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        
        loso_bal_acc.append(bal_acc)
        loso_f1.append(f1)
        
    print(f"-> Fine Test Subject {test_subject} | Balanced Acc: {bal_acc*100:.2f}% | F1-Score: {f1:.4f} | Tempo: {time.time()-start_time:.1f}s")

print("\n" + "="*60)
print(f"RISULTATI FINALI LOSO Epilessia (Latent Alignment)")
print(f"Balanced Accuracy Media: {np.mean(loso_bal_acc)*100:.2f}% ± {np.std(loso_bal_acc)*100:.2f}%")
print(f"F1-Score Medio:          {np.mean(loso_f1):.4f} ± {np.std(loso_f1):.4f}")
print("="*60)

Utilizzando il device: cpu
Pesi applicati alla Loss: Background=0.0504, Crisi=1.9496

=== Inizio Validazione LOSO su Epilessia (SeizureConvNet + Latent Alignment) ===

-> Addestramento per Test Subject: 1


KeyboardInterrupt: 